# Comparison across output resolutions

Section 4.2 (*Accuracy, cost, and output resolution*). The DL-ROM needs one checkpoint per resolution; SLAE and LLAE answer at any $N_t$ from a single $K=16$ checkpoint at fixed horizon $T$. Both accuracy and cost are swept, so the flexibility the article claims is measured rather than asserted.


In [ ]:
import sys, os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, os.path.join('..', 'src'))

import glob
import re
import gc
import numpy as np
import torch
import torch.nn.functional as F_nn
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from tqdm.auto import tqdm

from laplace_surrogate.inference.pipeline import InferencePipeline

In [ ]:
# Article figure style; `run` prefixes the output filenames.
import figstyle
figstyle.setup(run='dlrom')       # titles=True pour garder les suptitle en interactif


## 1. Configuration

DL-ROM sweep resolutions and Laplace reference checkpoints. All listed `K=16` variants are included: `_ll` (LLAE, learnable poles), `_ol` (SLAE, offline contour) and the prescribed-contour baseline.


In [ ]:
# DL-ROM emits only its native Nt, so the sweep needs one checkpoint per point.
NT_LIST   = [150, 75, 50, 30, 15]          # checkpoints DL-ROM disponibles
NT_FULL   = 150                            # horizon complet (nb de pas natif du dataset)

# Output resolutions for the cost-versus-Nt sweep. The Laplace pipelines emit
# any Nt from a single
#    checkpoint K=16 (predict(..., n_frames=Nt) / generate(Nt=...)) — horizon T conservé.
NT_GRID   = [15, 30, 50, 75, 150]

# Laplace references: every listed variant is included.
#    _ll = pôles appris (LLAE, g0.0) / _ol = contour optimisé (SLAE, g0.0)
#    nue  = basis fixe « classique » (g0.01, sans suffixe)
LLAE_REFS = [
    'LLAEModel__llae_ld64_K16_g0.0_ll__t*',   # pôles appris
    'LLAEModel__llae_ld64_K16_g0.01__t*',     # basis fixe « classique » (g0.01)
]
SLAE_REFS = [
    'SLAEModel__slae_ld64_K16_g0.0_ol__t*',   # contour optimisé hors-ligne
    'SLAEModel__slae_ld64_K16_g0.01__t*',     # basis fixe « classique » (g0.01)
]

POD_REF_MEDIAN = 8.91      # médiane L2rel (%) du meilleur POD surrogate (article §8)

DATA_PATH  = '../dataset/ch4_rotated.npy'
DOE_PATH   = '../dataset/doe_rotated.npy'
SPLIT_PATH = '../dataset/split.npz'
CKPT_DIR   = '../checkpoints'
CACHE_DIR  = 'cache_dlrom'
N          = 128
DT         = 1.0

os.makedirs(CACHE_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Dataset


In [ ]:
# Raw field data (mmap)
U_raw = np.load(DATA_PATH, mmap_mode='r')          # (ns, Nt, H, W)
ns, Nt_full, H, W = U_raw.shape
print(f'Dataset: {ns} sims, Nt={Nt_full}, {H}×{W}')
assert Nt_full == NT_FULL, f'Attendu Nt={NT_FULL}, trouvé {Nt_full}'

# Physical parameters θ = (k, A, C)
doe = np.load(DOE_PATH)
theta_np = np.stack([doe['k'], doe['A'], doe['C']], axis=1).astype(np.float32)  # (ns, 3)
print(f'θ shape: {theta_np.shape}  min={theta_np.min():.3f}  max={theta_np.max():.3f}')

# Train/test split
split     = np.load(SPLIT_PATH)
test_idx  = split['test_idx'].tolist()
train_idx = [i for i in range(ns) if i not in set(test_idx)]
print(f'Train: {len(train_idx)}  Test: {len(test_idx)}')

In [ ]:
def load_u(idx: int) -> np.ndarray:
    '''Load and resize U to (Nt, N, N) float32 (full 150-step horizon).'''
    u = U_raw[idx].astype(np.float32)
    if u.shape[-1] != N:
        u_t = torch.from_numpy(u).unsqueeze(1)
        u_t = F_nn.interpolate(u_t, size=(N, N), mode='bilinear', align_corners=False)
        u   = u_t.squeeze(1).numpy()
    return u

## 3. Checkpoint discovery

Naming conventions (`.pt` or `.ckpt`):

- DL-ROM: `DLROMModel__dlrom_ld64_Nt{Nt}__t*`, $N_t \in \{150,75,50,30,15\}$
- LLAE: `llae_ld64_K16` — `g0.0_ll` (learnable poles) and `g0.01` (prescribed)
- SLAE: `slae_ld64_K16` — `g0.0_ol` (offline contour) and `g0.01` (prescribed)

Missing patterns are logged and skipped.


In [ ]:
def _glob_both(stem_pattern: str) -> list:
    '''Glob a stem pattern over both .pt and .ckpt, sorted.'''
    hits = []
    for ext in ('.pt', '.ckpt'):
        hits += glob.glob(os.path.join(CKPT_DIR, stem_pattern + ext))
    return sorted(set(hits))


def _collect_refs(patterns: list, label: str) -> list:
    '''Return every existing checkpoint among the listed patterns, deduplicated and
    in pattern order. Missing patterns are logged and skipped.'''
    found, seen = [], set()
    for pat in patterns:
        hits = _glob_both(pat)
        if hits:
            for h in hits:                       # tous les hits (ex. -v1, -v2)
                if h not in seen:
                    seen.add(h); found.append(h)
                    print(f'  [{label}] retenu   : {os.path.basename(h)}')
        else:
            print(f'  [{label}] absent   : {pat}(.pt|.ckpt)')
    if not found:
        print(f'  [{label}] AUCUNE référence trouvée → famille ignorée dans les figures.')
    return found


# ── DL-ROM sweep ─────────────────────────────────────────────────────────────
print('DL-ROM sweep :')
dlrom_paths = {}    # Nt_nominal → path
for nt in NT_LIST:
    hits = _glob_both(f'DLROMModel__dlrom_ld64_Nt{nt}__t*')
    if hits:
        dlrom_paths[nt] = hits[-1]
        print(f'  [Nt={nt:>3}] {os.path.basename(hits[-1])}')
    else:
        print(f'  [Nt={nt:>3}] absent   : DLROMModel__dlrom_ld64_Nt{nt}__t*(.pt|.ckpt)')

# ── Références Laplace (toutes les variantes listées) ────────────────────────
print('\nRéférences Laplace :')
llae_refs = _collect_refs(LLAE_REFS, 'LLAE')
slae_refs = _collect_refs(SLAE_REFS, 'SLAE')

# ── Table de travail : name → path (ordre : DL-ROM par Nt ↓, puis réfs) ──────
ckpt_paths = {}
for nt in sorted(dlrom_paths, reverse=True):
    p = dlrom_paths[nt]
    ckpt_paths[os.path.splitext(os.path.basename(p))[0]] = p
for p in (*llae_refs, *slae_refs):
    ckpt_paths[os.path.splitext(os.path.basename(p))[0]] = p

print(f'\n{len(ckpt_paths)} checkpoint(s) à évaluer :')
for n in ckpt_paths:
    print('  ', n)
if not ckpt_paths:
    print('\n*** Aucun checkpoint surrogate trouvé. '
          'Ce notebook doit tourner sur le host GPU distant (voir préconditions). ***')

## 4. Palette and labels

Shared palette from `figstyle`, plus a colormap indexed by $N_t$ for the DL-ROM sweep.


In [ ]:
PALETTE = dict(figstyle.MODEL_COLORS)   # palette partagee — voir notebooks/figstyle.py

# Colormap indexed by Nt for the DL-ROM sweep (small Nt = low cost = light).
DLROM_CMAP = plt.get_cmap('viridis')

def dlrom_color(nt_eff: int, nt_list=None) -> tuple:
    '''Couleur d'un point DL-ROM selon son Nt effectif (léger→foncé).'''
    nts = sorted(nt_list or NT_LIST)
    frac = (nts.index(nt_eff) / max(len(nts) - 1, 1)) if nt_eff in nts else 0.5
    return DLROM_CMAP(0.15 + 0.7 * frac)


# AE stem Laplace, e.g. 'llae_ld64_K16_g0.1_ll' / 'slae_ld64_K16_g0.1_ol'
_AE_STEM_PAT = re.compile(
    r'^(?P<ae>slae|llae|lslae)_ld(?P<ld>\d+)_K(?P<K>\d+)_g(?P<g>[0-9.]+)(?P<ae_suf>.*)$'
)
# AE stem DL-ROM, e.g. 'dlrom_ld64_Nt75'
_AE_STEM_DLROM_PAT = re.compile(r'^dlrom_ld(?P<ld>\d+)_Nt(?P<Nt>\d+)$')

# Surrogate ckpt, e.g. 'DLROMModel__dlrom_ld64_Nt75__t4h2' or
# 'LLAEModel__llae_ld64_K16_g0.1_ll__t4h2[-v1]'
_SURR_PAT = re.compile(
    r'^(?P<cls>[A-Za-z]+Model)__(?P<ae_stem>.+?)__'
    r't(?P<n_trunk>\d+)h(?P<n_head>\d+)(?P<suffix>.*?)(?:-v(?P<ver>\d+))?$'
)


def _fmt_g(g: str) -> str:
    f = float(g)
    return str(int(f)) if f == int(f) else str(f)


def _pretty_name(name: str) -> str:
    '''Nom lisible sur deux lignes. Gère DL-ROM (dlrom_ld*_Nt*) et Laplace.'''
    m = _SURR_PAT.match(name)
    if not m:
        return name
    ae_stem = m.group('ae_stem')

    # ── DL-ROM ───────────────────────────────────────────────────────────────
    dm = _AE_STEM_DLROM_PAT.match(ae_stem)
    if dm:
        nt = int(dm.group('Nt'))
        return (f"DL-ROM  d={dm.group('ld')}, Nt={nt}\n"
                f"t{m.group('n_trunk')}h{m.group('n_head')}  ({nt} têtes)")

    # ── Laplace (SLAE / LLAE) ────────────────────────────────────────────────
    sm = _AE_STEM_PAT.match(ae_stem)
    if not sm:
        return name
    cls   = m.group('cls')
    label = 'LLAE' if cls.startswith('LLAE') else 'SLAE'
    head   = f"{label}  d={sm.group('ld')}, K={sm.group('K')}"
    ae_suf = sm.group('ae_suf').strip('_')
    head  += f' [{ae_suf}]' if ae_suf else f", γ={_fmt_g(sm.group('g'))}"
    return head + '\n' + f"t{m.group('n_trunk')}h{m.group('n_head')}"


def _family(model_type: str) -> str:
    return {'DLROMModel': 'DL-ROM', 'LLAEModel': 'LLAE', 'SLAEModel': 'SLAE'}.get(
        model_type, model_type)



# ── Style commun des figures (marqueur/couleur/label, EN) ────────────────────
def _laplace_basis(name):
    m  = _SURR_PAT.match(name)
    sm = _AE_STEM_PAT.match(m.group('ae_stem')) if m else None
    suf = sm.group('ae_suf').strip('_') if sm else ''
    if suf == 'll': return 'learned', 'learned poles'
    if suf == 'ol': return 'learned', 'optimized contour'
    return 'fixed', 'fixed contour'

def _mk(name):
    r = results[name]; mt = r['model_type']; c = PALETTE.get(mt, 'gray')
    if mt == 'DLROMModel':
        return dict(c=c, marker='o', size=110, face=c, edge='k', lw=0.8,
                    tag=f"DL-ROM $N_t$={r['Nt_eff']}")
    basis, bl = _laplace_basis(name); fam = _family(mt)
    if basis == 'learned':
        return dict(c=c, marker='*', size=340, face=c, edge='k', lw=0.9, tag=f"{fam} ({bl})")
    return dict(c=c, marker='D', size=150, face='white', edge=c, lw=1.8, tag=f"{fam} ({bl})")

def _legend_handle(name):
    m = _mk(name)
    return Line2D([0], [0], marker=m['marker'], color='w',
                  markerfacecolor=(m['face'] if m['face'] != 'white' else 'white'),
                  markeredgecolor=m['edge'], markeredgewidth=m['lw'],
                  markersize=13 if m['marker'] == '*' else 8, label=m['tag'])

def _pointwise_names(src):
    """DL-ROM Nt=150 uniquement + toutes les réfs Laplace."""
    return [n for n in src
            if results[n]['model_type'] != 'DLROMModel' or results[n]['Nt_eff'] == NT_FULL]


## 5. Evaluation — native-grid metric

Each checkpoint is read with its own temporal stride $\kappa$, and predictions are compared against the ground truth subsampled on the same grid, without interpolation. DL-ROM is therefore judged on its $N_{t,\mathrm{eff}}$ steps and the Laplace pipelines ($\kappa = 1$) on all 150.

A second metric, obtained by interpolating a coarse output back to 150 steps, is still computed and cached for format compatibility but is not used: output resolution is carried by the $N_t$ column, not by the metric.


In [ ]:
def resample_to_horizon(U_pred: np.ndarray, kappa: int, n_full: int = NT_FULL) -> np.ndarray:
    '''Resample (Nt_eff, N, N) to (n_full, N, N) by linear interpolation in time at the
    sampled positions np.arange(0, n_full, kappa), holding the last value beyond the
    final sample.'''
    if kappa == 1:
        return U_pred                                   # déjà 150 pas (Laplace / DL-ROM-150)
    sample_t = np.arange(0, n_full, kappa)              # (Nt_eff,)  ex. κ=10 → [0,10,…,140]
    assert U_pred.shape[0] == len(sample_t), (
        f'Nt_eff={U_pred.shape[0]} incohérent avec arange(0,{n_full},{kappa})={len(sample_t)}')
    target_t = np.arange(n_full)                        # 0..149
    idx = np.clip(np.searchsorted(sample_t, target_t, side='right') - 1,
                  0, len(sample_t) - 2)                 # segment [idx, idx+1]
    t0, t1 = sample_t[idx], sample_t[idx + 1]
    w = np.clip((target_t - t0) / (t1 - t0), 0.0, 1.0)  # >1 hors-borne → tenue (clip à 1)
    U0, U1 = U_pred[idx], U_pred[idx + 1]
    return U0 + (U1 - U0) * w[:, None, None]


def _l2rel(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(b)
    return float(np.linalg.norm(a - b) / denom * 100.0) if denom >= 1e-12 else np.nan


def _cost_meta(ckpt_path: str) -> dict:
    '''Cost metadata for a checkpoint, without any forward pass: loads the model on CPU
    and reads κ, Nt_eff, n_heads, n_surr and model_type.'''
    pipe   = InferencePipeline.from_checkpoint(ckpt_path, device='cpu')
    kappa  = int(pipe.ckpt.get('t_stride', 1))
    n_surr = int(sum(p.numel() for p in pipe.model.surrogate.parameters()))
    n_head = int(getattr(pipe.model.surrogate, 'K', pipe.ckpt.get('Nt', NT_FULL)))
    Nt_eff = int(pipe.ckpt.get('Nt', NT_FULL))
    mtype  = pipe.model_type
    pipe.model.cpu(); del pipe; gc.collect()
    return dict(kappa=kappa, Nt_eff=Nt_eff, n_heads=n_head, n_surr=n_surr, model_type=mtype)


def eval_from_l2rel(ckpt_path: str, l2rel: np.ndarray) -> dict:
    '''Résultat pour un modèle Laplace (κ=1) à partir d'un l2rel déjà calculé
    (cache_surr) : l2_grid = l2_horizon = l2rel — car κ=1 ⇒ pas de sous-échantillonnage
    ni de ré-interpolation, donc _l2rel(U_pred, U_full[::1]) == _l2rel(U_pred, U_full).
    Métadonnées via _cost_meta (chargement CPU seul, AUCUN forward sur le test set).'''
    meta = _cost_meta(ckpt_path)
    assert meta['kappa'] == 1, 'réutilisation cache_surr réservée aux modèles Laplace κ=1'
    l2 = np.asarray(l2rel, dtype=np.float64)
    return dict(l2_grid=l2, l2_horizon=l2.copy(), **meta)


def eval_dlrom(ckpt_path: str, idx_list: list, device='cpu', batch_size: int = 32) -> dict:
    '''Evaluate a checkpoint. Returns dict(l2_grid, l2_horizon, kappa, Nt_eff, n_heads,
    n_surr, model_type).'''
    pipe  = InferencePipeline.from_checkpoint(ckpt_path, device=device)
    name  = os.path.basename(ckpt_path)
    kappa = int(pipe.ckpt.get('t_stride', 1))
    mtype = pipe.model_type
    # Surrogate MLP parameters only: N_surr = N_trunk + K*N_head.
    n_surr  = int(sum(p.numel() for p in pipe.model.surrogate.parameters()))
    n_heads = int(getattr(pipe.model.surrogate, 'K', pipe.ckpt.get('Nt', NT_FULL)))

    l2_grid, l2_horizon = [], []
    for start in tqdm(range(0, len(idx_list), batch_size), desc=name, leave=False):
        batch_idx   = idx_list[start:start + batch_size]
        theta_batch = theta_np[batch_idx]              # (B, 3)
        U_pred = pipe.predict(theta_batch)             # (B, Nt_eff, N, N)
        for j, idx in enumerate(batch_idx):
            U_full = load_u(idx)                       # (150, N, N)
            # Native grid: ground truth subsampled onto the model's own time grid.
            l2_grid.append(_l2rel(U_pred[j], U_full[::kappa]))
            # (b) iso-horizon : prédiction ré-interpolée à 150 pas vs champ plein
            l2_horizon.append(_l2rel(resample_to_horizon(U_pred[j], kappa), U_full))
        if device == 'cuda':
            torch.cuda.empty_cache()

    Nt_eff = int(pipe.ckpt.get('Nt', U_pred.shape[1]))
    pipe.model.cpu(); del pipe
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return dict(
        l2_grid    = np.array(l2_grid),
        l2_horizon = np.array(l2_horizon),
        kappa      = kappa,
        Nt_eff     = Nt_eff,
        n_heads    = n_heads,
        n_surr     = n_surr,
        model_type = mtype,
    )

## 6. Evaluation of all checkpoints

Per-checkpoint cache invalidated on checkpoint mtime. For a Laplace reference ($\kappa = 1$) the surrogate cache already holds the same quantity, so it is reused and only the cost metadata is reloaded; everything else is computed on the GPU.


In [ ]:
SURR_CACHE_DIR = 'cache_surr'   # surrogate-notebook cache, reused when available

def _cache_path(name: str) -> str:
    return os.path.join(CACHE_DIR, f'{name}.npz')

def _cache_valid(cache_file: str, ckpt_path: str) -> bool:
    if not os.path.exists(cache_file):
        return False
    try:
        data = np.load(cache_file, allow_pickle=True)
        return float(data['ckpt_mtime']) == os.path.getmtime(ckpt_path)
    except Exception as e:
        print(f'Failed to load cache {cache_file}: {e}')
        return False

def _surr_l2rel(name: str, ckpt_path: str):
    '''Relative error from the surrogate cache if present and current, else None.
    For a Laplace model at κ=1 this equals both grid and horizon metrics.'''
    f = os.path.join(SURR_CACHE_DIR, f'{name}.npz')
    if not os.path.exists(f):
        return None
    try:
        d = np.load(f, allow_pickle=True)
        if float(d['ckpt_mtime']) != os.path.getmtime(ckpt_path):
            return None
        return d['l2rel']
    except Exception:
        return None

def _save_cache(name: str, ckpt_path: str, r: dict):
    np.savez(
        _cache_path(name),
        l2_grid    = r['l2_grid'],
        l2_horizon = r['l2_horizon'],
        kappa      = np.int64(r['kappa']),
        Nt_eff     = np.int64(r['Nt_eff']),
        n_heads    = np.int64(r['n_heads']),
        n_surr     = np.int64(r['n_surr']),
        model_type = str(r['model_type']),
        ckpt_mtime = np.float64(os.path.getmtime(ckpt_path)),
    )

def _load_cache(name: str) -> dict:
    d = np.load(_cache_path(name), allow_pickle=True)
    return dict(
        l2_grid    = d['l2_grid'],
        l2_horizon = d['l2_horizon'],
        kappa      = int(d['kappa']),
        Nt_eff     = int(d['Nt_eff']),
        n_heads    = int(d['n_heads']),
        n_surr     = int(d['n_surr']),
        model_type = str(d['model_type']),
    )


# Order of preference: own cache, then the surrogate cache for Laplace
# references, then a full GPU evaluation.
results = {}   # name → dict (voir eval_dlrom)

for name, ckpt_path in ckpt_paths.items():
    cache_file = _cache_path(name)
    if _cache_valid(cache_file, ckpt_path):
        r = _load_cache(name)
        print(f'[cache]   {name}  [{r["model_type"]}]')
    else:
        # Reuse the surrogate cache for Laplace references only: at κ>1 the two
        # metrics differ and are not stored there.
        l2r = None if name.startswith('DLROMModel') else _surr_l2rel(name, ckpt_path)
        if l2r is not None:
            print(f'[reuse cache_surr] {name}')
            try:
                r = eval_from_l2rel(ckpt_path, l2r)   # aucun forward sur le test set
                _save_cache(name, ckpt_path, r)
            except Exception as e:
                print(f'   reuse KO ({e}) → recalcul GPU'); l2r = None
        if l2r is None:
            print(f'[compute] {name}')
            try:
                r = eval_dlrom(ckpt_path, test_idx, device=device)
                _save_cache(name, ckpt_path, r)
            except Exception as e:
                print(f'   ERROR: {e}')
                continue
    results[name] = r
    g, h = r['l2_grid'], r['l2_horizon']
    print(f'   κ={r["kappa"]:>2}  Nt_eff={r["Nt_eff"]:>3}  heads={r["n_heads"]:>3}  '
          f'N_surr={r["n_surr"]/1e6:5.2f}M  |  '
          f'iso-grille med={np.nanmedian(g):5.2f}% p90={np.nanpercentile(g,90):5.2f}%  |  '
          f'iso-horizon med={np.nanmedian(h):5.2f}% p90={np.nanpercentile(h,90):5.2f}%')

print(f'\nDone. {len(results)} checkpoint(s) évalué(s).')

## 8. Surrogate size

$N_{\mathrm{surr}} = N_{\mathrm{trunk}} + K \cdot N_{\mathrm{head}}$ counts the surrogate MLP alone, excluding the decoder: about 23.5M at $N_t = 150$ for DL-ROM against 3.5M at $K = 16$ for the Laplace pipelines.


In [ ]:
def _compression(r: dict) -> float:
    return NT_FULL / r['n_heads']

if results:
    print(f"{'checkpoint':<40} {'famille':<8} {'κ':>3} {'Nt_eff':>6} {'têtes':>6} "
          f"{'N_surr(M)':>10} {'compr.×':>8}")
    print('-' * 90)
    for name, r in sorted(results.items(), key=lambda kv: -kv[1]['n_heads']):
        print(f"{name:<40} {_family(r['model_type']):<8} {r['kappa']:>3} {r['Nt_eff']:>6} "
              f"{r['n_heads']:>6} {r['n_surr']/1e6:>10.2f} {_compression(r):>8.1f}")
else:
    print('Pas de résultats — checkpoints absents (host GPU distant requis).')

## 9. Inference cost

Latency, FLOPs, peak memory and energy at batch size 1, measured by `notebooks/benchmark.py`. The axes need not agree — see Dehghani et al. (2022) on the efficiency misnomer — which is why all four are reported.


In [ ]:
import benchmark   # methodologie commune a tous les notebooks

bench = {}

for name, ckpt_path in tqdm(ckpt_paths.items(), desc='inference benchmark'):
    if name not in results:
        continue
    try:
        pipe = InferencePipeline.from_checkpoint(ckpt_path, device=device)
        tmean = torch.tensor(pipe.ckpt['theta_mean'], dtype=torch.float32, device=device)
        tstd  = torch.tensor(pipe.ckpt['theta_std'],  dtype=torch.float32, device=device)
        th1 = (torch.tensor(theta_np[test_idx[:1]], dtype=torch.float32, device=device) - tmean) / tstd

        def gen1():
            with torch.no_grad():
                pipe.model.generate(th1)

        bench[name] = benchmark.measure(gen1, device, name=name)

        pipe.model.cpu(); del pipe
        if device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        print(f'{name}  ERROR: {e}')

print('\nDone.')


## 10. Benchmark — latency (bar chart)

Per-sample latency at `bs=1` (10 warm-up / 50 CUDA-Event runs), DL-ROM at $N_t=150$.

In [ ]:
# One figure per metric.
if bench and results:
    _names = [n for n in _pointwise_names(results) if n in bench]
    _b = {n: bench[n] for n in _names}

    def _bench_label(t):
        return _mk(t)['tag']

    def _bench_color(t):
        return PALETTE.get(results[t]['model_type'], 'gray')

    benchmark.table(_b, label_of=_bench_label)
    benchmark.figures(_b, label_of=_bench_label, color_of=_bench_color, device=device)
else:
    print('No bench — skipped.')


## 11. Summary table

One row per DL-ROM resolution, with the Laplace references below. Output resolution is carried by the $N_t$ column.


In [ ]:
if results:
    def _row(name, r):
        g    = r['l2_grid']
        lat  = bench.get(name, {}).get('lat_med', np.nan)
        return dict(
            fam    = _family(r['model_type']),
            Nt     = r['Nt_eff'],
            kappa  = r['kappa'],
            compr  = _compression(r),
            surrM  = r['n_surr'] / 1e6,
            lat    = lat,
            g_med  = np.nanmedian(g),  g_p90 = np.nanpercentile(g, 90),
        )

    dl_rows  = [(n, _row(n, r)) for n, r in results.items() if r['model_type'] == 'DLROMModel']
    lap_rows = [(n, _row(n, r)) for n, r in results.items()
                if r['model_type'] in ('LLAEModel', 'SLAEModel')]
    dl_rows.sort(key=lambda kv: -kv[1]['Nt'])
    lap_rows.sort(key=lambda kv: kv[1]['fam'])

    hdr = (f"{'modèle':<14}{'Nt':>4}{'κ':>4}{'compr×':>8}{'N_surr(M)':>11}"
           f"{'lat(ms)':>9}{'grille med':>12}{'grille p90':>12}")
    print(hdr); print('-' * len(hdr))
    def _pr(tag, d):
        lat = f"{d['lat']:>9.1f}" if np.isfinite(d['lat']) else f"{'—':>9}"
        print(f"{tag:<14}{d['Nt']:>4}{d['kappa']:>4}{d['compr']:>8.1f}{d['surrM']:>11.2f}"
              f"{lat}{d['g_med']:>11.2f}%{d['g_p90']:>11.2f}%")
    for _, d in dl_rows:
        _pr(f"DL-ROM Nt{d['Nt']}", d)
    print('-' * len(hdr))
    for _, d in lap_rows:
        _pr(f"{d['fam']}-K16", d)
    print(f"{'POD surrogate':<14}{'—':>4}{'—':>4}{'—':>8}{'—':>11}{'—':>9}"
          f"{POD_REF_MEDIAN:>11.2f}%{'—':>11}")

    # DataFrame optionnel (si pandas dispo)
    try:
        import pandas as pd
        rows = ([dict(model=f"DL-ROM Nt{d['Nt']}", **d) for _, d in dl_rows] +
                [dict(model=f"{d['fam']}-K16", **d) for _, d in lap_rows])
        df = pd.DataFrame(rows)[['model', 'Nt', 'kappa', 'compr', 'surrM', 'lat',
                                 'g_med', 'g_p90']]
        display(df.round(2))
    except Exception:
        pass
else:
    print('Pas de résultats — table sautée.')

## 12. Accuracy vs inference cost

Median relative $L^2$ error (iso-grid, no interpolation) against five cost axes at fixed
150-step output: surrogate **parameters**, **latency**, **peak GPU memory**, **FLOPs**,
**energy**. Bottom-left is best. ★ = learned basis, ◆ = fixed basis; DL-ROM shown at
$N_t=150$ (same task).

In [ ]:
# Accuracy vs cost — 5 panels (DL-ROM at Nt=150 only). Shared legend.
_PW_AXES = [
    ('params',  'Surrogate parameters (M)', 1e-6),
    ('lat_med', 'Latency (ms)',             1.0),
    ('mem_mb',  'Peak GPU memory (MB)',     1.0),
    ('flops',   'FLOPs (G)',                1e-9),
    ('energy_j','Energy (mJ)',              1e3),
]

def _cost(n, k, sc):
    if k == 'params':
        return results[n]['n_surr'] * sc
    v = bench.get(n, {}).get(k, np.nan)
    return v * sc if np.isfinite(v) else np.nan

if bench and results:
    pw = [n for n in _pointwise_names(results) if n in bench]
    fig, axs = plt.subplots(2, 3, figsize=(16, 9)); axs = axs.reshape(-1)
    for ax, (k, lab, sc) in zip(axs, _PW_AXES):
        for n in pw:
            x = _cost(n, k, sc)
            if not np.isfinite(x): continue
            m = _mk(n)
            ax.scatter(x, np.nanmedian(results[n]['l2_grid']), s=m['size'], marker=m['marker'],
                       facecolors=m['face'], edgecolors=m['edge'], lw=m['lw'], zorder=5)
        ax.axhline(POD_REF_MEDIAN, ls='--', color='gray', lw=1.0, zorder=1)
        ax.set_xlabel(lab); ax.set_ylabel('Relative $L^2$ error (%)')
        ax.grid(alpha=0.3); ax.margins(x=0.16, y=0.16)
    # 6e case = légende commune
    axs[5].axis('off')
    handles = [_legend_handle(n) for n in pw]
    handles.append(Line2D([0], [0], ls='--', color='gray', lw=1.0,
                          label=f'POD surrogate ({POD_REF_MEDIAN:.2f}%)'))
    axs[5].legend(handles=handles, loc='center', fontsize=11, frameon=False,
                  title='Model', title_fontsize=12)
    gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
    fig.suptitle(f'Accuracy vs inference cost (bs=1, 150-step output) — {gpu}',
                 fontweight='bold', fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    print('No bench/results — skipped.')
figstyle.save("cost_axes")


## 13. Relative cost vs full DL-ROM ($N_t=150$ = 1)

Same 150-step output, cost normalized to DL-ROM $N_t=150$ on each axis. Bars `< 1` are
cheaper than the baseline (log scale).

In [ ]:
# Cost relative to DL-ROM Nt=150 (=1) on: params, latency, memory, FLOPs, energy.
def _dl150():
    for n, r in results.items():
        if r['model_type'] == 'DLROMModel' and r['Nt_eff'] == NT_FULL: return n
    return None

_ref = _dl150()
if _ref is not None and bench:
    axes_def = [
        ('params',  lambda n: results[n]['n_surr']),
        ('latency', lambda n: bench.get(n, {}).get('lat_med', np.nan)),
        ('memory',  lambda n: bench.get(n, {}).get('mem_mb',  np.nan)),
        ('FLOPs',   lambda n: bench.get(n, {}).get('flops',   np.nan)),
        ('energy',  lambda n: bench.get(n, {}).get('energy_j', np.nan)),
    ]
    ref_v = {lab: g(_ref) for lab, g in axes_def}
    labs = [lab for lab, _ in axes_def if np.isfinite(ref_v[lab]) and ref_v[lab] > 0]
    laps = [n for n in _pointwise_names(results)
            if results[n]['model_type'] in ('LLAEModel', 'SLAEModel') and n in bench]
    laps.sort(key=lambda n: (results[n]['model_type'], _laplace_basis(n)[0]))

    if laps and labs:
        x = np.arange(len(labs)); w = 0.8 / max(len(laps), 1)
        fig, ax = plt.subplots(figsize=(1.7 * len(labs) + 3, 5.4))
        getter = dict(axes_def)
        for i, n in enumerate(laps):
            m = _mk(n)
            ratios = [getter[lab](n) / ref_v[lab] for lab in labs]
            xi = x + (i - (len(laps) - 1) / 2) * w
            ax.bar(xi, ratios, w * 0.95, color=m['c'],
                   alpha=0.5 if m['face'] == 'white' else 0.9, edgecolor=m['c'], lw=1.5,
                   hatch='//' if m['face'] == 'white' else None, label=m['tag'])
        ax.axhline(1.0, color='crimson', lw=1.4, ls='--')
        ax.text(len(labs) - 0.5, 1.05, 'DL-ROM $N_t$=150 (=1)', ha='right',
                color='crimson', fontsize=9, fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(labs)
        ax.set_ylabel('Cost relative to DL-ROM $N_t$=150 (log)'); ax.set_yscale('log')
        ax.grid(axis='y', alpha=0.3, which='both')
        ax.legend(fontsize=9, ncol=2, loc='upper center')
        fig.tight_layout();
        plt.show()
    else:
        print('Missing Laplace refs / cost axes — skipped.')
else:
    print('DL-ROM Nt=150 or bench missing — skipped.')
figstyle.save("cost_relative")


## 14. Inference cost vs output resolution $N_t$

Laplace pipelines output any $N_t$ from a single $K{=}16$ checkpoint; DL-ROM needs one model
per $N_t$. Cost (params, FLOPs, latency, memory, energy) of one inference vs the requested
number of output steps. **SLAE is flat** (decodes $K{=}16$ frames, decoupled from $N_t$);
LLAE and DL-ROM grow as $O(N_t)$ and overlap in FLOPs (decoder-bound); LLAE wins on params
(fixed surrogate).

In [ ]:
# Inference cost versus output resolution Nt, at batch size 1.
#   DL-ROM  : point unique à son Nt natif (n_frames ignoré).
#   LLAE/SLAE: one checkpoint, generate(Nt=nt) swept over NT_GRID.
cost_nt = {}   # name -> {nt: dict(flops, lat_ms, mem_mb, energy_j)}
_cost_errors = {}

def _measure(gen, name, nt):
    d = {'flops': np.nan, 'lat_ms': np.nan, 'mem_mb': np.nan, 'energy_j': np.nan}
    # 1) sanity : l'inférence tourne-t-elle ? (sinon on logue l'erreur RÉELLE)
    try:
        gen()
    except Exception as e:
        import traceback
        if name not in _cost_errors:
            _cost_errors[name] = f'{type(e).__name__}: {e}'
            print(f'  !! {name} Nt={nt} generate KO -> {type(e).__name__}: {e}')
            traceback.print_exc()
        return d
    try:    d['flops'] = _flops(gen)
    except Exception as e: print(f'  !! {name} Nt={nt} FLOPs KO -> {e}')
    if device == 'cuda':
        try:    d['lat_ms'] = float(np.median(_time_gpu(gen, 5, 30)))
        except Exception as e: print(f'  !! {name} Nt={nt} lat KO -> {e}')
        try:    d['mem_mb'] = _peak_mem_mb(gen)
        except Exception: pass
        if pynvml is not None:
            try:    d['energy_j'] = _energy_gpu(gen, _NVML_H)[0]
            except Exception: pass
    return d

for name, ckpt_path in tqdm(ckpt_paths.items(), desc='coût vs Nt'):
    if name not in results:
        continue
    mt = results[name]['model_type']
    try:
        pipe = InferencePipeline.from_checkpoint(ckpt_path, device=device)
        tmean = torch.tensor(pipe.ckpt['theta_mean'], dtype=torch.float32, device=device)
        tstd  = torch.tensor(pipe.ckpt['theta_std'],  dtype=torch.float32, device=device)
        tn = (torch.tensor(theta_np[test_idx[:1]], dtype=torch.float32, device=device) - tmean) / tstd
        is_lap = mt in ('LLAEModel', 'SLAEModel')
        nts = list(NT_GRID) if is_lap else [results[name]['Nt_eff']]
        cost_nt[name] = {}
        for nt in nts:
            if is_lap:
                def gen(nt=nt, m=pipe.model, tn=tn):
                    with torch.no_grad(): m.generate(tn, Nt=nt)
            else:
                def gen(m=pipe.model, tn=tn):
                    with torch.no_grad(): m.generate(tn)
            cost_nt[name][nt] = _measure(gen, name, nt)
        pipe.model.cpu(); del pipe
        if device == 'cuda': torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        print(f'{name}  ERROR: {e}')

# ── Résumé : FLOPs mesurés par modèle (NaN = mesure échouée) ─────────────────
print('\nFLOPs (G) mesurés par Nt :')
for name in cost_nt:
    fl = [cost_nt[name][nt].get('flops', np.nan) / 1e9 for nt in sorted(cost_nt[name])]
    fl_s = ' '.join(f'{v:5.1f}' if np.isfinite(v) else '  nan' for v in fl)
    print(f'  {name:<44} {fl_s}')
if _cost_errors:
    print('\nErreurs generate :', _cost_errors)
print('\nDone.')

In [ ]:
# Cost vs output resolution Nt — 5 panels, shared legend, EN.
_NT_AXES = [
    ('params',  'Surrogate parameters (M)', 1e-6),
    ('flops',   'FLOPs (G)',                1e-9),
    ('lat_ms',  'Latency (ms)',             1.0),
    ('mem_mb',  'Peak GPU memory (MB)',     1.0),
    ('energy_j','Energy (mJ)',              1e3),
]

def _ntval(n, nt, k, sc):
    if k == 'params':
        return results[n]['n_surr'] * sc
    v = cost_nt[n].get(nt, {}).get(k, np.nan)
    return v * sc if np.isfinite(v) else np.nan

if cost_nt:
    common = [n for n in cost_nt if cost_nt[n]]
    laps = [n for n in common if results[n]['model_type'] in ('LLAEModel', 'SLAEModel')]
    dros = [n for n in common if results[n]['model_type'] == 'DLROMModel']
    axes = [(k, lab, sc) for (k, lab, sc) in _NT_AXES
            if k == 'params' or any(np.isfinite(cost_nt[n][nt].get(k, np.nan))
                                    for n in common for nt in cost_nt[n])]
    fig, axs = plt.subplots(2, 3, figsize=(16, 9)); axs = axs.reshape(-1)
    for ax, (k, lab, sc) in zip(axs, axes):
        # DL-ROM : fin pointillé SOUS tout le monde (zorder 2)
        dls = sorted((results[n]['Nt_eff'], _ntval(n, results[n]['Nt_eff'], k, sc)) for n in dros)
        if dls:
            xd, yd = zip(*dls)
            ax.plot(xd, yd, '--', color=PALETTE['DLROMModel'], lw=1.3, zorder=2)
        # Laplace : lignes pleines (SLAE zorder 3, LLAE zorder 4)
        for n in laps:
            m = _mk(n); nts = sorted(cost_nt[n])
            z = 4 if results[n]['model_type'] == 'LLAEModel' else 3
            ax.plot(nts, [_ntval(n, nt, k, sc) for nt in nts], '-', color=m['c'], lw=2.2, zorder=z,
                    marker=m['marker'], markersize=13 if m['marker'] == '*' else 7,
                    markerfacecolor=(m['c'] if m['face'] != 'white' else 'white'),
                    markeredgecolor=(m['c'] if m['face'] == 'white' else 'k'))
        # DL-ROM drawn as open rings on top so the LLAE line stays visible through them.
        if dls:
            ax.scatter(xd, yd, s=130, facecolors='none', edgecolors=PALETTE['DLROMModel'],
                       linewidths=2.0, zorder=6)
        ax.set_xlabel('Output resolution $N_t$'); ax.set_ylabel(lab)
        ax.grid(alpha=0.3); ax.set_xticks(NT_GRID)

    # 6e case = legende commune (ordre : LLAE, SLAE, DL-ROM)
    axs[5].axis('off')
    handles = [_legend_handle(n) for n in laps]
    handles.append(Line2D([0], [0], ls='--', color=PALETTE['DLROMModel'], marker='o',
                          markerfacecolor='none', markeredgecolor=PALETTE['DLROMModel'],
                          markeredgewidth=2.0, label='DL-ROM (one model / $N_t$)'))
    axs[5].legend(handles=handles, loc='center', fontsize=11, frameon=False,
                  title='Model', title_fontsize=12)
    gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
    fig.suptitle(f'Inference cost vs output resolution — SLAE flat, LLAE fixed params — {gpu}',
                 fontweight='bold', fontsize=13.5)
    fig.tight_layout();
    plt.show()
else:
    print('cost_nt empty — skipped.')
figstyle.save("cost_vs_nt")


## Accuracy versus output resolution

The DL-ROM is trained at each resolution; the Laplace pipelines are merely queried
there. The comparison is therefore between a specialised model and one evaluated away
from its training grid, which is the point rather than a confound.

`rescale_reg` recalibrates the Tikhonov terms when the output resolution changes
($\alpha_t \sim 1/\Delta t$, $\lambda \sim \Delta t$). Both settings are reported:
if the recalibration matters, resolution flexibility is not free, and that is a result
in itself.


In [ ]:
def eval_laplace_at_nt(ckpt_path, idx_list, n_frames, rescale_reg=True,
                       device='cuda', batch_size=32):
    """Relative L2 error of a Laplace pipeline queried at `n_frames` output steps.

    The horizon T is preserved, so the emitted grid is arange(0, NT_FULL, kappa) with
    kappa = NT_FULL // n_frames — the same grid the DL-ROM checkpoints are trained on,
    which makes the two directly comparable.
    """
    pipe  = InferencePipeline.from_checkpoint(ckpt_path, device=device)
    kappa = NT_FULL // n_frames

    # Highest angular frequency carried by the contour, kept for the Nyquist floor.
    try:
        omega_max[ckpt_path] = float(pipe.model.laplace.s_list.imag.abs().max())
    except Exception:
        pass
    errs  = []
    for start in range(0, len(idx_list), batch_size):
        batch = idx_list[start:start + batch_size]
        U_pred = pipe.predict(theta_np[batch], n_frames=n_frames,
                              rescale_reg=rescale_reg)
        for j, idx in enumerate(batch):
            errs.append(_l2rel(U_pred[j], load_u(idx)[::kappa]))
        if device == 'cuda':
            torch.cuda.empty_cache()
    pipe.model.cpu(); del pipe
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return np.array(errs)


# Full test set: the sweep is quoted alongside the other tables, and the first 200
# instances happen to be markedly harder than average (4.06% against 5.94% median for
# LLAE), so a subset would not be comparable.
nt_idx = test_idx

laplace_refs = {n: p for n, p in ckpt_paths.items() if not n.startswith('DLROMModel')}

# Highest angular frequency carried by each contour, read straight from the stored
# poles. Computed here rather than inside the evaluation loop, which is skipped
# whenever the cache is warm and would leave the sampling floor undrawn.
omega_max = {}
for _n, _p in laplace_refs.items():
    _sd = torch.load(_p, map_location='cpu', weights_only=False).get('state_dict', {})
    if 'model.laplace.s_im' in _sd:
        omega_max[_n] = float(_sd['model.laplace.s_im'].abs().max())
print('omega_max:', {k: round(v, 4) for k, v in omega_max.items()})

nt_acc = {}  # (name, rescale) -> {Nt: median l2rel}
for name, path in laplace_refs.items():
    for rescale in (True, False):
        per_nt = {}
        for nt in NT_GRID:
            cache = os.path.join(CACHE_DIR, f'nt{nt}_{"r" if rescale else "n"}_{name}.npz')
            if os.path.exists(cache) and \
               float(np.load(cache)['ckpt_mtime']) == os.path.getmtime(path):
                e = np.load(cache)['l2rel']
            else:
                e = eval_laplace_at_nt(path, nt_idx, nt, rescale, device=device)
                np.savez(cache, l2rel=e,
                         ckpt_mtime=np.float64(os.path.getmtime(path)))
            per_nt[nt] = float(np.median(e))
        nt_acc[(name, rescale)] = per_nt
        tag = 'rescaled' if rescale else 'raw'
        print(f'{_mk(name)["tag"]:<28} {tag:<9} ' +
              '  '.join(f'Nt={nt}: {per_nt[nt]:5.2f}%' for nt in NT_GRID))


In [ ]:
OUTLIER = 20.0   # above this a configuration is past its sampling floor

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)

for ax, rescale in zip(axes, (True, False)):
    # Points past a configuration's sampling floor are off by one to two orders of
    # magnitude. They are dropped rather than clipped: a clipped line still runs to
    # the top of the frame and reads as a trend, which it is not.
    for name in laplace_refs:
        per_nt = nt_acc[(name, rescale)]
        xs = [nt for nt in NT_GRID if per_nt[nt] <= OUTLIER]
        ax.plot(xs, [per_nt[nt] for nt in xs], marker='o', ms=5, lw=1.8,
                color=PALETTE.get(results[name]['model_type'], 'gray'),
                label=_mk(name)['tag'])

    # DL-ROM: one checkpoint per resolution, plotted as a reference.
    dl = sorted((results[n]['Nt_eff'], np.median(results[n]['l2_grid']))
                for n in results if n.startswith('DLROMModel'))
    if dl:
        ax.plot([d[0] for d in dl], [d[1] for d in dl], marker='s', ms=6, lw=1.8,
                ls='--', color=PALETTE['DLROMModel'], label='DL-ROM (trained per $N_t$)')

    # Sampling floor: below Nt = NT_FULL * omega_max / pi the fastest encoded
    # frequency has fewer than two samples per period and is aliased.
    # One floor per contour: the optimised contours dilate their imaginary parts,
    # so they encode higher frequencies and need a finer output grid than the
    # prescribed ones. Drawing a single floor would hide exactly that trade-off.
    for _n, _w in sorted(omega_max.items(), key=lambda kv: kv[1]):
        nt_min = NT_FULL * _w / np.pi
        col = PALETTE.get(results[_n]['model_type'], 'gray')
        ax.axvline(nt_min, ls=':', lw=1.4, color=col, alpha=0.8, zorder=1)
        ax.annotate(f'$N_t={nt_min:.0f}$', (nt_min, 0.97),
                    xycoords=('data', 'axes fraction'), xytext=(3, 0),
                    textcoords='offset points', fontsize=7, color=col,
                    va='top', ha='left', rotation=90)

    # Log y: the sub-floor values span three orders of magnitude above the rest,
    # and a linear axis flattens the 4-10% range that carries the result.
    ax.set_xscale('log')
    ax.set_xticks(NT_GRID)
    ax.set_xticklabels([str(n) for n in NT_GRID])
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.set_xlabel('Output resolution $N_t$'
                  + ('   (regularisation rescaled)' if rescale
                     else '   (regularisation unchanged)'))
    ax.grid(alpha=0.35, which='both')

axes[0].set_ylabel(figstyle.LABELS['l2rel'])
axes[0].legend(fontsize=8, loc='upper right')
# Not phrased as "below the sampling floor": LLAE with learnable poles sits below
# its own floor at Nt=30 and is still plotted, at 9.8%. Crossing a floor degrades
# gracefully for some contours and catastrophically for others, so the criterion
# that governs what is drawn is the value, not the crossing.
axes[0].annotate(r'points above 20\% omitted' if mpl.rcParams['text.usetex']
                 else 'points above 20% omitted',
                 (0.02, 0.94), xycoords='axes fraction', fontsize=8, color='0.35',
                 va='top', ha='left')
fig.tight_layout()
plt.show()
figstyle.save("accuracy_vs_nt")
